In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

In [ ]:
df = pd.read_csv('NVDA_hourly_last_2_years.csv')
df = df.drop(index=[0, 1])
df = df.rename(columns={'Price': 'Datetime'})   

In [ ]:
df['Datetime'] = pd.to_datetime(df['Datetime'])
df['Close'] = pd.to_numeric(df['Close'])
df['High'] = pd.to_numeric(df['High'])
df['Low'] = pd.to_numeric(df['Low'])
df['Open'] = pd.to_numeric(df['Open'])
df['Volume'] = pd.to_numeric(df['Volume'])

In [ ]:
df['Target'] = df['Close'].shift(-1)
df['Change'] = ((df['Close'] - df['Close'].shift(1)) / df['Close'].shift(1)) * 100

In [ ]:
from indicators.momentum import ROC, RSI

from indicators.trend import ADX, EMA, MACD, SMA

from indicators.volatility import ATR, Bollinger_Bands

from indicators.volume import OBV, VWAP

### Momentum Indicators

In [ ]:
df['ROC'] = ROC(df['Close'])

df['RSI'] = RSI(df['Close'])

### Trend Indicators

In [ ]:
df['ADX'] = ADX(df['High'], df['Low'], df['Close'])

df['EMA_10'] = EMA(df['Close'], 10)
df['EMA_20'] = EMA(df['Close'], 20)
df['EMA_50'] = EMA(df['Close'], 50)
df['EMA_100'] = EMA(df['Close'], 100)

df['SMA_10'] = SMA(df['Close'], 10)
df['SMA_20'] = SMA(df['Close'], 20)
df['SMA_50'] = SMA(df['Close'], 50)
df['SMA_100'] = SMA(df['Close'], 100)

df['MACD_Line'], df['MACD_Signal'] = MACD(df['Close'])

### Volatility Indicators

In [ ]:
df['BB_upper'], df['BB_lower'] = Bollinger_Bands(df['Close'], 20, 2)

df['ATR'] = ATR(df['High'], df['Low'], df['Close'])

### Volume Indicators

In [ ]:
df['OBV'] = OBV(df['Close'], df['Volume'])

df['VWAP'] = VWAP(df['Close'], df['Volume'])

### Feature Transformation

In [ ]:
ma_columns = [col for col in df.columns if 'SMA' in col or 'EMA' in col]
for col in ma_columns:
    df[f'Dist_{col}'] = (df['Close'] - df[col]) / df[col]

if 'BB_upper' in df.columns and 'BB_lower' in df.columns:
    df['BB_Percent'] = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])

cols_to_drop = ma_columns + ['BB_upper', 'BB_lower']
df = df.drop(columns=cols_to_drop)

print(f"Transformed features and dropped {len(cols_to_drop)} raw columns. {cols_to_drop}")

### Feature Selection & Dimensionality Reduction

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

df_clean = df.dropna()

exclude_cols = ['Datetime', 'Target', 'Change', 'Close', 'Open', 'High', 'Low', 'Volume']
feature_cols = [col for col in df_clean.columns if col not in exclude_cols]

X = df_clean[feature_cols]
y = df_clean['Target']

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Training evaluator model...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Calculating permutation importance (this may take a moment)...")
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

print("\n--- Top 10 Most Valuable Features ---")
print(importance_df.head(10))

surviving_features = importance_df[importance_df['Importance'] > 0.0001]['Feature'].tolist()

final_cols = ['Datetime', 'Close', 'Target', 'Change'] + surviving_features
df_final = df_clean[final_cols]

print(f"\nFeature selection complete! Reduced feature space from {len(feature_cols)} down to {len(surviving_features)}.]")